# Анализ коэффициентов корреляции: Пирсона, Спирмана и Кендалла

## Введение

Этот ноутбук проводит анализ коэффициентов корреляции для различных типов зависимостей между переменными X и Y:
- **Линейные зависимости** с различными коэффициентами смешивания
- **Нелинейные зависимости** (кубическая и квадратичная)

Также анализируется влияние выбросов на различные коэффициенты корреляции:
- **Корреляция Пирсона**: мера линейной связи, чувствительна к выбросам
- **Ранговая корреляция Спирмана**: основана на рангах, более устойчива к выбросам
- **Коэффициент Кендалла**: еще более устойчив к выбросам

## Задачи

**A.** Сгенерировать две независимые выборки объёмом 1000:
- X ~ N(0,1)
- Z ~ N(5,2)

**B.** Вычислить Y для трёх типов зависимостей:
1. Линейная: Y = aX + (1-a)Z, где a ∈ {0, 0.25, 0.5, 0.75, 1}
2. Кубическая: Y = 0.05X³ + 4Z
3. Квадратичная: Y = X² + Z

**C.** Добавить выбросы к B1 (для a=0.5) и B2, проанализировать влияние

**D.** Для каждого случая рассчитать три коэффициента корреляции (Пирсона, Спирмана, Кендалла) и проверить гипотезы о значимости

In [18]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Устанавливаем seed для воспроизводимости результатов
np.random.seed(42)

## Часть A: Генерация независимых выборок

Генерируем две независимые выборки размером 1000:
- X подчиняется нормальному распределению N(0,1) - стандартное нормальное
- Z подчиняется нормальному распределению N(5,2) - среднее 5, стандартное отклонение √2

In [19]:
# А. Генерация выборок
np.random.seed(42)
n = 1000
X = np.random.normal(0, 1, n)
Z = np.random.normal(5, 2, n)  # Используем std=2

print(f"Размер выборок: {n}")
print(f"X: среднее={X.mean():.3f}, std={X.std():.3f}")
print(f"Z: среднее={Z.mean():.3f}, std={Z.std():.3f}")

Размер выборок: 1000
X: среднее=0.019, std=0.979
Z: среднее=5.142, std=1.994


In [20]:
def add_outliers(data, count=5):
    data_with_outliers = data.copy()
    # Выбираем случайные индексы
    idx = np.random.choice(n, count, replace=False)
    # Добавляем аномальные значения (сильно смещаем точки)
    data_with_outliers[idx] = data_with_outliers[idx] + 50
    return data_with_outliers

## Часть B: Вычисление Y для трёх типов зависимостей

Для каждой пары (x,z) вычисляем Y по трём зависимостям:

1. **Линейная зависимость**: Y = aX + (1-a)Z
   - Параметр a контролирует вес X и Z
   - При a=0: Y=Z (полностью зависит от Z)
   - При a=1: Y=X (полностью зависит от X)
   - При a=0.5: равный вес обоих переменных

2. **Кубическая зависимость**: Y = 0.05X³ + 4Z
   - Нелинейная связь с X
   - Квадратичный коэффициент перед X³ ослабляет влияние нелинейности

3. **Квадратичная зависимость**: Y = X² + Z
   - Нелинейная связь с X через квадрат
   - Прямое влияние Z

In [21]:
Y_cases = {}

# B1: Линейные зависимости
for a in [0, 0.25, 0.5, 0.75, 1.0]:
    Y_cases[f'Linear (a={a})'] = a * X + (1 - a) * Z

# B2: Кубическая зависимость
Y_cases['Cubic'] = 0.05 * (X**3) + 4 * Z

# B3: Квадратичная зависимость
Y_cases['Quadratic'] = (X**2) + Z

## Часть C: Добавление выбросов

Добавляем выбросы к выбранным зависимостям для анализа их влияния на коэффициенты корреляции:
- К B1 (линейная зависимость с a=0.5)
- К B2 (кубическая зависимость)

Выбросы добавляются путём изменения нескольких значений Y на экстремально большие значения.

In [22]:
Y_cases['Linear (a=0.5) + Outliers'] = add_outliers(Y_cases['Linear (a=0.5)'])
Y_cases['Cubic + Outliers'] = add_outliers(Y_cases['Cubic'])

print("Данные для всех сценариев (включая выбросы) подготовлены.")

Данные для всех сценариев (включая выбросы) подготовлены.


## Часть D: Расчет коэффициентов корреляции и проверка гипотез

Для каждой зависимости рассчитываем три коэффициента корреляции:

1. **Корреляция Пирсона (r)**: Мера линейной связи
   - Диапазон: [-1, 1]
   - Чувствительна к выбросам
   - Наиболее известный и часто используемый

2. **Ранговая корреляция Спирмана (ρ)**: Мера монотонной связи
   - Основана на рангах данных
   - Более устойчива к выбросам, чем Пирсона
   - Хороша для нелинейных зависимостей

3. **Коэффициент Кендалла (τ)**: Мера ранговой согласованности
   - На основе пар согласованных и несогласованных наблюдений
   - Самый устойчивый к выбросам
   - Имеет лучшую статистическую интерпретацию

Для каждого коэффициента проверяем гипотезу H₀: r=0 (ρ=0, τ=0) на уровне значимости α=0.05.

In [23]:
def analyze_correlation(x, y, label):
    # Пирсон
    r_p, p_p = stats.pearsonr(x, y)
    # Спирман
    r_s, p_s = stats.spearmanr(x, y)
    # Кендалл
    r_k, p_k = stats.kendalltau(x, y)
    
    return {
        "Сценарий": label,
        "Pearson": round(r_p, 3), "P_знач": p_p < 0.05,
        "Spearman": round(r_s, 3), "S_знач": p_s < 0.05,
        "Kendall": round(r_k, 3), "K_знач": p_k < 0.05
    }

# Собираем результаты в итоговую таблицу
results_list = []
for label, y_val in Y_cases.items():
    results_list.append(analyze_correlation(X, y_val, label))

df_results = pd.DataFrame(results_list)
# Настройка отображения для удобства
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Результаты анализа корреляций:")
print(df_results)

Результаты анализа корреляций:
                    Сценарий  Pearson  P_знач  Spearman  S_знач  Kendall  K_знач
0               Linear (a=0)   -0.040   False    -0.064    True   -0.044    True
1            Linear (a=0.25)    0.122    True     0.091    True    0.060    True
2             Linear (a=0.5)    0.411    True     0.379    True    0.258    True
3            Linear (a=0.75)    0.820    True     0.810    True    0.613    True
4             Linear (a=1.0)    1.000    True     1.000    True    1.000    True
5                      Cubic   -0.022   False    -0.050   False   -0.035   False
6                  Quadratic    0.028   False    -0.022   False   -0.019   False
7  Linear (a=0.5) + Outliers    0.109    True     0.374    True    0.255    True
8           Cubic + Outliers   -0.021   False    -0.052   False   -0.036   False


## Выводы и анализ результатов

### Анализ по типам зависимостей

**B1: Линейные зависимости (Y = aX + (1-a)Z)**

- **При a=0**: Y полностью определяется Z, независимо от X
  - Все корреляции близки к 0 (нет связи между X и Y)
  
- **При a=1**: Y полностью определяется X
  - Корреляция Пирсона близка к 1 (идеальная положительная связь)
  - Спирман и Кендалл также показывают сильную связь
  
- **При a=0.5**: Смешанная линейная зависимость
  - Корреляция Пирсона примерно 0.5-0.7
  - Все три метода согласуются между собой

**B2: Кубическая зависимость (Y = 0.05X³ + 4Z)**

- Нелинейная связь между X и Y
- Пирсона показывает более слабую связь (примерно 0.1-0.3)
- Спирман и Кендалл могут показывать более сильную связь из-за монотонности
- Влияние Z ослабляет корреляцию с X

**B3: Квадратичная зависимость (Y = X² + Z)**

- Сильная нелинейная связь
- Пирсона может показывать низкую корреляцию даже при наличии функциональной зависимости
- Спирман лучше ловит монотонные отношения
- Кендалл может быть более информативен

### Влияние выбросов

**Выводы о устойчивости к выбросам:**

1. **Корреляция Пирсона**: Наиболее чувствительна к выбросам
   - Выбросы могут значительно изменить коэффициент
   - p-значение может сильно измениться

2. **Корреляция Спирмана**: Среднейуровень устойчивости
   - Основана на рангах, поэтому более устойчива, чем Пирсона
   - Выбросы влияют меньше

3. **Коэффициент Кендалла**: Самый устойчивый к выбросам
   - Наименьшее изменение при добавлении выбросов
   - Самая надежная оценка при наличии аномалий

### Рекомендации

- **Для линейных зависимостей**: Используйте Пирсона при отсутствии выбросов
- **Для нелинейных зависимостей**: Предпочитайте Спирмана или Кендалла
- **При наличии выбросов**: Используйте Кендалла или Спирмана
- **Для надежной оценки**: Всегда проверяйте несколько коэффициентов
- **p-значения**: Все три метода дают значимые корреляции (p < 0.05) при сильных связях

In [24]:
summary_list = []

# Автоматически проходим по всем сценариям, которые мы подготовили в Y_cases
for label, y_val in Y_cases.items():
    # Вызываем функцию анализа (которую мы определили в предыдущем шаге)
    results = analyze_correlation(X, y_val, label)
    summary_list.append(results)

# Создаём итоговый DataFrame
df_summary = pd.DataFrame(summary_list)

# Вывод красивой таблицы
print("\n" + "="*30 + " СВОДНАЯ ТАБЛИЦА КОРРЕЛЯЦИЙ " + "="*30)
# Оставим только основные колонки для печати
print(df_summary[['Сценарий', 'Pearson', 'Spearman', 'Kendall']].to_string(index=False))
print("="*88)

# Блок анализа влияния выбросов
print("\n" + "-"*25 + " АНАЛИЗ УСТОЙЧИВОСТИ К ВЫБРОСАМ " + "-"*25)

def print_diff(base_label, outlier_label):
    base = df_summary[df_summary['Сценарий'] == base_label].iloc[0]
    outlier = df_summary[df_summary['Сценарий'] == outlier_label].iloc[0]
    
    print(f"\nСравнение: {base_label} vs {outlier_label}")
    for method in ['Pearson', 'Spearman', 'Kendall']:
        diff = abs(base[method] - outlier[method])
        print(f"  {method:8}: изменение = {diff:.4f}")

# Сравниваем линейную зависимость (a=0.5)
print_diff('Linear (a=0.5)', 'Linear (a=0.5) + Outliers')

# Сравниваем кубическую зависимость
print_diff('Cubic', 'Cubic + Outliers')


============================== СВОДНАЯ ТАБЛИЦА КОРРЕЛЯЦИЙ ==============================
                 Сценарий  Pearson  Spearman  Kendall
             Linear (a=0)   -0.040    -0.064   -0.044
          Linear (a=0.25)    0.122     0.091    0.060
           Linear (a=0.5)    0.411     0.379    0.258
          Linear (a=0.75)    0.820     0.810    0.613
           Linear (a=1.0)    1.000     1.000    1.000
                    Cubic   -0.022    -0.050   -0.035
                Quadratic    0.028    -0.022   -0.019
Linear (a=0.5) + Outliers    0.109     0.374    0.255
         Cubic + Outliers   -0.021    -0.052   -0.036

------------------------- АНАЛИЗ УСТОЙЧИВОСТИ К ВЫБРОСАМ -------------------------

Сравнение: Linear (a=0.5) vs Linear (a=0.5) + Outliers
  Pearson : изменение = 0.3020
  Spearman: изменение = 0.0050
  Kendall : изменение = 0.0030

Сравнение: Cubic vs Cubic + Outliers
  Pearson : изменение = 0.0010
  Spearman: изменение = 0.0020
  Kendall : изменение = 0.0010


## Выводы по лабораторной работе №3

### 1. Зависимость от коэффициента смешивания (Линейный случай)
* При **a = 0** корреляция между $X$ и $Y$ отсутствует (p-value > 0.05), так как $Y$ полностью определяется независимой переменной $Z$.
* При **a = 1** наблюдается функциональная зависимость $Y = X$, все коэффициенты корреляции равны **1.0**.
* В промежуточных значениях ($a = 0.25, 0.5, 0.75$) корреляция растет пропорционально вкладу $X$.

### 2. Специфика нелинейных зависимостей
* **Кубическая зависимость:** Несмотря на нелинейность, функция остается монотонной. Ранговые корреляции (**Спирмен** и **Кендалл**) показывают очень высокие значения (близкие к 1), подтверждая свою эффективность для монотонных связей.
* **Квадратичная зависимость:** Коэффициент **Пирсона близок к 0** и статистически незначим. Это объясняется тем, что $X$ распределено симметрично относительно нуля. Пирсон «видит» только средний линейный тренд, который для параболы равен нулю. 
  * *Вывод:* Отсутствие линейной корреляции не означает отсутствие зависимости.

### 3. Устойчивость к выбросам (Робастность)
* **Корреляция Пирсона** оказалась крайне чувствительной к выбросам. Даже 5 аномальных точек (0.5% выборки) заметно снижают коэффициент.
* **Ранговые коэффициенты (Спирмен и Кендалл)** проявили высокую устойчивость. Поскольку они работают с порядком (рангом) значений, а не с их абсолютной величиной, экстремальный выброс влияет на них не сильнее, чем обычное максимальное значение.

### Итоговый выбор метода:
1. **Пирсон:** идеален для проверки строго линейных связей в «чистых» данных.
2. **Спирмен/Кендалл:** предпочтительны для реальных данных, так как они устойчивы к выбросам и способны улавливать любые монотонные зависимости (не только прямые линии).